### 📘 学习打卡

##### 请执行下方 cell，点击窗口中的“开始学习”按钮开始本次学习，完成学习后请点击“结束学习”按钮。开始和结束时间都会记录到统一日志文件中，帮助对你的学习情况进行分析。

In [1]:
from projects.clock_in import create_session_widgets
display(create_session_widgets("projects/cfg.log"))

# 上下文无关文法与推导
---

## 概览

本部分将介绍编译原理中的核心概念：**上下文无关文法（CFG）**。编译器的语法分析阶段正是基于 CFG 来理解源代码的层次结构。掌握 CFG 的基本概念和推导的过程，是理解编译器（词法分析器模块）构造方法的基础。

## 学习目标

1. 理解 **上下文无关文法（CFG）** 的基本概念及其四元式形式化定义。
2. 掌握对于自然语言描述的符号串集合，**设计对应的 CFG** 的方法。
3. 理解 **推导** 的过程，特别是最左推导。
4. 理解 **语法分析树** 的构造及其与二义性、运算符优先级和结合率的关系。

## 1. 上下文无关文法基本概念

本部分将介绍如何利用上下文无关文法来描述编程语言的语法结构，包括文法的形式化定义、符号约定，以及如何通过文法设计来描述复杂的符号串集合。

### 1.1 概述与语法定义

在编译过程中，词法分析器将源代码分解为一个个单词（token），而语法分析器则需要分析这些单词之间的关系，判断它们是否构成了合法的语法结构。为了描述这种结构，我们使用 **上下文无关文法（Context-Free Grammar, CFG）**。

巴科斯-瑙尔范式通常用于表示 CFG。例如，C语言中的 `if` 语句可以这样描述：

`stmt -> if ( expr ) stmt else stmt`

在这个规则中：
* `if`, `else`, `(`, `)` 是**终结符（terminal）**，即词法分析器输出的单词，它们是源程序中出现的语言的基本符号，不可再分。
* `expr`, `stmt` 是**非终结符（nonterminal）**，表示语法变量或语法范畴，它们代表了更复杂的语法结构，并不在源程序中直接出现，需要通过对应规则进一步展开。
* `->` 读作“定义为”或“推导为”，它表示左部的非终结符可展开为右部的符号串。这种规则称为**产生式**。

### 1.2 形式化定义：四元式与符号串基础

为了严格地定义一个文法，我们需要引入一些形式语言理论的基本概念：

1. **字母表与符号串**：字母表 $\Sigma$ 是一个有穷的符号集合。符号串是字母表中符号构成的有穷序列。空字 $\epsilon$ 是不含任何符号的序列。
2. **闭包与正则闭包**：
    * $V^*$（闭包）：表示 $V$ 中符号串进行任意有限次（包括0次）连接构成的集合。即 $V^* = V^0 \cup V^1 \cup V^2 \dots$，其中 $V^0 = \{\epsilon\}$。$V^i$ 表示 $V$ 中符号串进行 $i$ 次连接（拼接）得到的所有可能符号串的集合。
    * $V^+$（正闭包）：表示 $V$ 中符号串进行至少1次连接构成的集合。即 $V^+ = V V^*$。
    * 由上述定义可知：$V^* = V^+ \cup \{\epsilon\}$。

**上下文无关文法的四元式定义**：

一个 CFG 可以形式化地表示为一个四元式 $G = (V_T, V_N, S, P)$：
1. **$V_T$**：非空有限的**终结符集合**。
2. **$V_N$**：非空有限的**非终结符集合**。$V_T \cap V_N = \emptyset$。
3. **$S$**：**开始符号**，$S \in V_N$，它是文法的核心起点。
4. **$P$**：**产生式集合**。每个产生式形式为 $A \rightarrow \alpha$，其中 $A \in V_N$（左部），$\alpha \in (V_T \cup V_N)^*$（右部）。

**思考：**$(V_T \cup V_N)^*$ 表示什么含义？

#### 💡 思考题 1.3

**问题：**

> 1. 给定以下文法规则：
>
> `expr -> expr + digit | expr - digit | digit`
>
> `digit -> 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9`
>
> 请指出该文法的终结符集合 $V_T$ 和非终结符集合 $V_N$ 分别包含哪些元素？开始符号 $S$ 是什么？
> 
> 2. 设有符号串集合 $V = \{a, \epsilon\}$，请写出 $V^*$ 和 $V^+$ 的结果，并说明它们之间的关系。

请运行下面的 cell，将会打开 **AI 助教窗口**，你在其中向 AI 助教回答两个问题，AI 助教会评判你的解答，并引导你修正错误答案。你可以与 AI 助教讨论、寻求提示，但不要要求其直接给出答案。

In [2]:
%run projects/problem_ai_helper.py 1.3

D:\code_warehouse\Curriculum\Compiler Systems\cfg\projects\problem_ai_helper.py:218: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.input_box.on_submit(self.send_message)


<IPython.core.display.Javascript object>

### 1.4 符号约定与 CFG 设计基础

在实际书写文法时，为了简洁，我们通常遵循一些常见的符号约定：
* **终结符**：数字、运算符、标点符号、**黑体字符串**（如 **if**, **while**）。
* **非终结符**：*斜体字符串*（如 *expr*, *stmt*）。
* **候选式合并**：如果多个产生式左部相同，可以合并为一个，右部用 `|` 分隔，表示“或”。

**完整示例**：

考虑 C 语言中的 `if-else` 语句，其语法结构为：if (expression) statement else statement。按照上述约定，我们可以写出如下文法规则：

*stmt* -> **if** ( *expr* ) *stmt* **else** *stmt*

在这个例子中，**if**、**else**、`(`、`)` 是终结符（黑体关键字或标点），*stmt* 和 *expr* 是非终结符（斜体）。

**CFG 的设计思想**：

设计 CFG 的过程，本质上是把我们头脑中清晰的语法概念转化为计算机可处理的符号规则，可以遵循以下步骤：

1. 首先是“人会做”
   - 我们自己先把要定义的语法概念“长什么模样”搞清楚
   - 例如，“if 语句”这个语法概念的模样是 **if关键字 括号包围的表达式 真分支语句 else关键字 假分支语句**
2. 然后是“让计算机会做”，即符号化：
   - 首先，为这个语法概念起个名字——通常是英文（缩写）。例如，if 语句可能起名 *if-stmt*。但，考虑到 if 语句是**语句**的一种，在语法分析中，它应该与其他语句同等对待，因此可以用 *stmt* 统一表示所有类型的语句
   - 其次，梳理“模样”中的各个组成部分是语法概念还是终结符，为它们起名字。如，**if**、**else**、`(`、`)`是终结符，真、假分支语句都是语法概念，都用 *stmt* 表示。
   - 将语法概念放在产生式左部，“模样”放在产生式右部，中间用 -> 连接。

### 1.5 CFG 设计练习：C 语言 for 语句

**问题：**

> 请为像下面这样的 C 语言 `for` 语句设计对应的上下文无关文法（CFG）：
> 
> `for ( i = 1 ; i <= 100 ; i++ ) sum = sum + i ;`
> 
> **提示**：请回顾 1.4 节介绍的“CFG 的设计思想”，并参考前面 `if` 语句的示例，按如下思路思考：
> 1. 一条 `for` 语句由几个部分组成？
> 2. 其中哪些是固定的终结符（如关键字、标点），哪些是需要进一步展开的语法概念（非终结符）？
> 3. 为“for 语句”这个语法概念起个什么名字？怎么为各组成部分起名字？
> 4. 怎么把它们串接起来形成完整的产生式？
> 5. 简单起见，只需设计一个产生式即可，无需为“for 语句”之外的语法概念设计对应产生式。

请执行下方 cell，输入你设计的产生式规则，并与 AI 助教讨论。

In [3]:
%run projects/problem_ai_helper.py 1.5

D:\code_warehouse\Curriculum\Compiler Systems\cfg\projects\problem_ai_helper.py:218: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.input_box.on_submit(self.send_message)


<IPython.core.display.Javascript object>

### 1.6 CFG 设计练习：C 语言变量声明

**问题：**

> 在 C 语言中，变量声明可以是不同类型、可以包含任意多个标识符，例如：
> 
> `int a;`
> 
> `float x, y, z;`
> 
> 请你设计一个 CFG 来描述这样的 C 语言变量声明语句。
> 
> **提示**：同样运用“CFG 的设计思想”，重点思考：
> 1. 可以将这类语句分解为哪两个主要部分？
> 2. 第一部分有几种“模样”？每种“模样”是终结符，还是非终结符？
> 3. 第二部分有几种“模样”？设计产生式描述所有“模样”的难点在哪里？（如果有无穷多种情况，需要用**递归**结构，回顾我们在之前理论课中学习的递归结构文法的设计——分为基本规则（最简单的情况）和递归规则（把问题规模变小））
> 4. 通过上面的提示，你应该能够猜出，在这个练习中，需要为多个语法概念设计对应的产生式，你可以采用“自顶向下”的方式。

请执行下方 cell，与 AI 助教讨论你的设计思路，最终设计出完整的CFG。

In [4]:
%run projects/problem_ai_helper.py 1.6

D:\code_warehouse\Curriculum\Compiler Systems\cfg\projects\problem_ai_helper.py:218: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.input_box.on_submit(self.send_message)


<IPython.core.display.Javascript object>

## 2. 推导

本部分将介绍如何从文法的开始符号出发，通过推导产生语言中的句子，并引入语法分析树来直观表示推导过程。同时，探讨文法的二义性以及如何通过文法设计来解决优先级和结合率问题。

### 2.1 推导与语言的概念

**推导** 是指从文法的开始符号 $S$ 出发，反复使用产生式（即用产生式的右部替换左部的非终结符），最终得到一个仅由终结符组成的单词串（称为**句子**）的过程。

例如，对于以下文法：

`expr -> expr + digit | expr - digit | digit`

`digit -> 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9`

表达式 `9 - 5 + 2` 的推导过程如下：

```
expr    => expr + digit           (P1: expr -> expr + digit)
        => expr - digit + digit   (P2: expr -> expr - digit)
        => digit - digit + digit  (P3: expr -> digit)
        => 9 - digit + digit      (P4: digit -> 9)
        => 9 - 5 + digit          (P4: digit -> 5)
        => 9 - 5 + 2              (P4: digit -> 2)
```

**思考**：对于给定的句子寻找它的推导过程时（这其实就是语法分析），在每一步，如何确定选择哪个候选式（如果被替换的非终结符有多个候选式的话）？——语法分析的难点！

**语言**：由文法 $G$ 所定义的语言 $L(G)$，就是从开始符号 $S$ 出发，经过有限步推导所能得到的**所有终结符串的集合**。

### 2.2 语法分析树

**语法分析树**（parse tree，简称语法树）是推导过程的图形化表示，它能清晰地展现句子的层次结构。构建语法树的规则如下：

1. 根节点标记为文法的**开始符号**。
2. 每个叶节点由一个终结符或 $\epsilon$ 标记。
3. 每个内部节点由一个非终结符标记。
4. 与推导过程的对应：若某步推导是应用产生式 $A \rightarrow X_1 X_2 \dots X_n$ 进行的替换，则将 $X_1, X_2, \dots, X_n$ 构建为节点 $A$ 的 $n$ 个孩子节点（从左到右）。
5. 对于产生式 $A \rightarrow \epsilon$，构建节点 $A$ 唯一的孩子节点 $\epsilon$。

**输出**：语法树叶节点从左至右排列构成的符号串，称为该树的**输出（yield）**，即开始符号推导出的句子。

对于 2.1 节中 `9 - 5 + 2` 的推导过程，如下图所示，其对应的语法树清晰地展示了表达式的结构。注意：推导时替换非终结符的不同顺序可能会生成相同的语法树。

<img src="./images/parse_tree_9-5+2.png" alt="9-5+2的语法树" style="width: 30%; display: block; margin-left: 0;">


### 2.3 二义性、优先级与结合率

#### 二义性
如果对一个文法，存在某个句子可以生成**两棵不同的语法树**，那么这个文法就是**二义性的**。

例如，对文法：

`expr -> expr + expr | expr - expr | 0 | 1 | ... | 9`

存在句子 `9 - 5 + 2`，它可以被解释为 `(9 - 5) + 2` 或 `9 - (5 + 2)`，从而产生两棵不同的语法树，如下图所示。二义性会导致编译器对程序结构的理解不确定。

<img src="./images/ambiguity_tree.png" alt="二义性语法树对比" style="width: 40%; display: block; margin-left: 0;">

不存在固定的算法来帮助我们消除二义性，只能**重新设计文法**。例如，对于算术表达式文法，我们需要在文法中体现下面两个性质来消除二义性。

#### 结合率
为了避免二义性，我们需要在文法中体现运算符的结合性质：
* **左结合**：如 `+, -, *, /`。`9-5-2` 应解释为 `(9-5)-2`。在文法中通常通过**左递归**（如 `expr -> expr - digit`，即，产生式有部第一个符号即为左部非终结符）来实现左结合。如下图所示，其语法树呈现“左倾”。

<img src="./images/left_assoc_tree.png" alt="左结合语法树" style="width: 15%; display: block; margin-left: 0;">

* **右结合**：如 `^, =`。在文法中通常通过**右递归**实现。如下图所示，其语法树呈现“右倾”。

<img src="./images/right_assoc_tree.png" alt="右结合语法树" style="width: 15%; display: block; margin-left: 0;">

#### 优先级
对于 `9 + 5 * 2`，我们期望 `*` 先于 `+` 计算。可以通过引入不同的非终结符表示不同优先级层次的运算来规定优先级。例如，表示乘/除运算（结果）的非终结符作为加/减运算的运算对象，从而体现出前者优先级高。

**无二义的表达式文法设计**：

`expr -> expr + term | expr - term | term`

`term -> term * factor | term / factor | factor`

`factor -> digit | ( expr )`

`digit -> 0 | 1 | 2 | 3 | ... | 9`

在这个文法中，`expr`、`term`、`factor` 代表了不同的优先级层次的运算，如下图所示，语法树中层次越低的非终结符（运算）优先级越高，层次越高的优先级越低。

<img src="./images/precedence_tree.png" alt="优先级语法树" style="width: 18%; display: block; margin-left: 0;">

### 2.4 动手练习：最左推导与语法树生成（含优先级与括号）

**问题：**

> 考虑以下无二义的表达式文法：
> 
> `expr -> expr + term | expr - term | term`
> 
> `term -> term * factor | term / factor | factor`
> 
> `factor -> digit | ( expr )`
> 
> `digit -> 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9`
> 
> 请写出句子 `7 - 3 * (4 + 6)` 的**最左推导**过程。

**要求：**
* 在下方的代码框中 **TODO** 位置输入推导过程，每一行输入一步推导。
* 输入格式参考第一行给出的第一步推导：推导符号必须使用 `=>`，`=>` 后面是这一步推导出的**完整符号串**（建议符号之间用空格分隔以获得最佳识别效果，程序已支持自动处理括号粘连），将这步推导过程放在一对双引号内，最后加上逗号（最后一步不加）。
* 例如：`expr => expr - term` 或 `=> term - term`。
* 输入完推导过程后，请执行这个 Cell，程序会自动验证你的推导步骤是否正确（必须是严格的最左推导，且能推导出目标句子），并为你**自动生成对应的语法树**。
* 如果遇到困难，可以运行下方的 AI 助教 cell 进行讨论。修改推导过程、重新保存/验证后，需重新执行 AI 助教 cell，以便基于新的解答进行讨论。

In [22]:
import sys
sys.path.append('projects')
from derivation_helper import validate_and_draw

# 文法的非终结符集合
non_terminals = {"expr", "term", "factor", "digit"}
# 目标句子
target = "7-3*(4+6)"

# 学生在此输入最左推导过程
# 【格式要求】每一行必须包含 "=>"，=> 之后是这一步推导出的完整符号串。
derivation_steps = [
    "expr => expr - term",
    "expr - term => term - term",
    "term - term => factor - term",
    "factor - term => digit - term",
    "digit - term => 7 - term",
    "7 - term => 7 - term * factor",
    "7 - term * factor => 7 - factor * factor",
    "7 - factor * factor => 7 - digit * factor",
    "7 - digit * factor => 7 - 3 * factor",
    "7 - 3 * factor => 7 - 3 * (expr)",
    "7 - 3 * (expr) => 7 - 3 * (expr + term)",
    "7 - 3 * (expr + term) => 7 - 3 * (term + term)",
    "7 - 3 * (term + term) => 7 - 3 * (factor + term)",
    "7 - 3 * (factor + term) => 7 - 3 * (digit + term)",
    "7 - 3 * (digit + term) => 7 - 3 * (4 + term)",
    "7 - 3 * (4 + term) => 7 - 3 * (4 + factor)",
    "7 - 3 * (4 + factor) => 7 - 3 * (4 + digit)",
    "7 - 3 * (4 + digit) => 7 - 3 * (4 + 6)"
    # TODO: 请继续输入推导步骤，直到最终推导出 "=> 7 - 3 * ( 4 + 6 )"
    
]

# 调用封装好的验证与绘图引擎
validate_and_draw(derivation_steps, non_terminals, target, "expr", "student_derivation_2.4.txt", "练习 2.4 (最左推导) - 学生输入")

✅ 你的推导步骤已保存。开始验证...

✅ 推导正确！最终生成句子: 7-3*(4+6)
📊 正在生成语法树...

expr
├── expr
│   └── term
│       └── factor
│           └── digit
│               └── 7
├── -
└── term
    ├── term
    │   └── factor
    │       └── digit
    │           └── 3
    ├── *
    └── factor
        ├── (
        ├── expr
        │   ├── expr
        │   │   └── term
        │   │       └── factor
        │   │           └── digit
        │   │               └── 4
        │   ├── +
        │   └── term
        │       └── factor
        │           └── digit
        │               └── 6
        └── )


In [23]:
%run projects/code_ai_helper.py 2.4

D:\code_warehouse\Curriculum\Compiler Systems\cfg\projects\code_ai_helper.py:351: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.input_box.on_submit(self.send_message)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 2.5 动手练习：后缀表达式文法的推导与语法树

**问题：**

> 考虑以下文法：
> 
> `S -> S S + | S S * | a`
> 
> 请写出句子 `aa+a*` 的**最左推导**过程，并观察生成的语法树。

**要求：**
* 在下方的代码框中 TODO 位置输入推导过程，每一行输入一步推导。
* 输入格式参考第一行给出的第一步推导：推导符号必须使用 `=>`，`=>` 后面是这一步推导出的**完整符号串**（建议符号之间用空格分隔），将这步推导过程放在一对双引号内，最后加上逗号（最后一步不加）。
* 例如：`S => S S *` 或 `=> a S *`。
* 完成推导过程输入后，运行 cell，程序会自动验证你的推导步骤并**自动生成对应的语法树**。
* 如果遇到困难，可以运行下方的 AI 助教 cell 进行讨论。

In [24]:
import sys
sys.path.append('projects')
from derivation_helper import validate_and_draw

# 文法的非终结符集合
non_terminals = {"S"}
# 目标句子
target = "aa+a*"

# 学生在此输入最左推导过程
derivation_steps = [
    "S => S S *",
    "S S * => S S + S *",
    "S S + S * => a S + S *",
    "a S + S * => a a + S *",
    "a a + S * => a a + a *",
    # TODO: 请继续输入推导步骤，直到最终推导出 "=> a a + a *"
    
]

# 调用封装好的验证与绘图引擎
validate_and_draw(derivation_steps, non_terminals, target, "S", "student_derivation_2.5.txt", "练习 2.5 (后缀推导) - 学生输入")

✅ 你的推导步骤已保存。开始验证...

✅ 推导正确！最终生成句子: aa+a*
📊 正在生成语法树...

S
├── S
│   ├── S
│   │   └── a
│   ├── S
│   │   └── a
│   └── +
├── S
│   └── a
└── *


In [25]:
%run projects/code_ai_helper.py 2.5

D:\code_warehouse\Curriculum\Compiler Systems\cfg\projects\code_ai_helper.py:351: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.input_box.on_submit(self.send_message)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. 总结

本 Notebook 介绍了编译原理中描述语法结构的核心工具——**上下文无关文法（CFG）**。我们学习了以下关键内容：

1. **文法的构成**：CFG 由终结符、非终结符、开始符号和产生式组成（四元式 $G = (V_T, V_N, S, P)$）。理解终结符与与非终结符的区别是进行文法设计的基础。
2. **文法设计**：掌握了将自然语言描述的语法规则转化为形式化文法的方法。无论是像 `for` 语句这样的顺序结构，还是像变量声明那样需要递归的结构，都可以通过合理的非终结符定义和产生式设计来描述。
3. **推导与语法树**：推导是从开始符号生成句子的过程。最左推导是语法分析中常用的一种规范推导方式。语法分析树直观地展示了句子的层次结构，且同一句子的不同语法树反映了文法的二义性。
4. **二义性的消除**：通过重新设计文法，引入不同层次的非终结符（如 `expr`、`term`、`factor`）来体现运算符的优先级，并通过左/右递归来体现结合律，从而消除文法的二义性。

掌握上下文无关文法及其推导过程，是理解编译器如何分析源代码语法结构的关键。在后续的课程中，我们将基于这些文法，学习如何手工编写语法分析器（如预测分析器）以及如何利用语法制导翻译技术生成中间代码。